In [1]:
import json
import math
from scipy.stats import pearsonr
import pandas as pd

# Evaluation Functions

In [2]:
def read_jsonl(file_path, task=3, data_type='pred'):
    key_name = {1: "Aspect_VA", 2: "Triplet", 3: 'Quadruplet'}
    output_key = key_name[task]
    input_key = key_name[3] if (data_type == 'gold' and task == 2) else key_name[task]
    
    data = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
            json_data = json.loads(line)
            entry = {
                'ID': json_data['ID'],
                'Text': json_data.get('Text', ''),
                'Aspect': json_data.get('Aspect', []),
            }
            
            quadruplets = json_data.get(input_key, [])
            if data_type == 'gold' and len(quadruplets) == 0:
                quadruplets = json_data.get(output_key, [])
            
            parsed_quadruplets = []
            for quad in quadruplets:
                parsed_quadruplets.append({
                    'Aspect': quad.get('Aspect', '').lower(),
                    'Category': quad.get('Category', '').lower(),
                    'Opinion': quad.get('Opinion', '').lower(),
                    'VA': quad['VA']
                })
            entry[output_key] = parsed_quadruplets
            data.append(entry)
    return data

In [3]:
def evaluate_task1(gold_data, pred_data, is_norm=True):
    gold_dict = {entry['ID']: entry for entry in gold_data}
    pred_dict = {entry['ID']: entry for entry in pred_data}
    
    gold_v, gold_a, pred_v, pred_a = [], [], [], []
    
    for key, value in gold_dict.items():
        gold_value = value["Aspect_VA"]
        pred_value = {entry['Aspect']: entry for entry in pred_dict[key]["Aspect_VA"]}
        
        for item in gold_value:
            gold_va = item['VA'].split("#")
            gold_v.append(float(gold_va[0]))
            gold_a.append(float(gold_va[1]))
            
            pred_va = pred_value[item['Aspect']]["VA"].split("#")
            pred_v.append(float(pred_va[0]))
            pred_a.append(float(pred_va[1]))
    
    pcc_v = pearsonr(pred_v, gold_v)[0]
    pcc_a = pearsonr(pred_a, gold_a)[0]
    
    gold_va = gold_v + gold_a
    pred_va = pred_v + pred_a
    
    mse = sum([(a - b)**2 for a, b in zip(gold_va, pred_va)]) / len(gold_v)
    rmse_va = math.sqrt(mse) / math.sqrt(128) if is_norm else math.sqrt(mse)
    
    return {'PCC_V': pcc_v, 'PCC_A': pcc_a, 'RMSE_VA': rmse_va}

In [4]:
def evaluate_task23(gold_data, pred_data, task=3):
    key_name = {2: "Triplet", 3: 'Quadruplet'}
    key = key_name[task]
    key_fields = ['Aspect', 'Opinion'] if task == 2 else ['Aspect', 'Opinion', 'Category']
    
    gold_dict = {entry['ID']: entry[key] for entry in gold_data}
    pred_dict = {entry['ID']: entry[key] for entry in pred_data}
    
    cTP_total = 0.0
    TP_cat = 0
    FP_cat = 0
    FN_cat = 0
    
    all_ids = set(gold_dict.keys()).union(set(pred_dict.keys()))
    
    for id_ in all_ids:
        gold_quads = gold_dict.get(id_, [])
        pred_quads = pred_dict.get(id_, [])
        matched_pred_num = 0
        
        for gold_quad in gold_quads:
            all_cTP_scores = []
            gold_key = tuple(gold_quad[f] for f in key_fields)
            
            for pred_quad in pred_quads:
                pred_key = tuple(pred_quad[f] for f in key_fields)
                
                if gold_key == pred_key:
                    gold_v, gold_a = map(float, gold_quad['VA'].split('#'))
                    pred_v, pred_a = map(float, pred_quad['VA'].split('#'))
                    
                    if not (1 <= pred_v <= 9 and 1 <= pred_a <= 9):
                        all_cTP_scores.append(0)
                        continue
                    
                    va_euclid = math.sqrt((pred_v - gold_v)**2 + (pred_a - gold_a)**2)
                    D_max = math.sqrt(128)
                    cTP_t = max(0.0, 1.0 - (va_euclid / D_max))
                    all_cTP_scores.append(cTP_t)
            
            if len(all_cTP_scores) > 1:
                FN_cat += 1
            elif len(all_cTP_scores) == 1:
                matched_pred_num += 1
                TP_cat += 1
                cTP_total += all_cTP_scores[0]
            else:
                FN_cat += 1
        
        FP_cat += (len(pred_quads) - matched_pred_num)
    
    cPrecision = cTP_total / (TP_cat + FP_cat) if (TP_cat + FP_cat) > 0 else 0.0
    cRecall = cTP_total / (TP_cat + FN_cat) if (TP_cat + FN_cat) > 0 else 0.0
    cF1 = 2 * cPrecision * cRecall / (cPrecision + cRecall) if (cPrecision + cRecall) > 0 else 0.0
    
    return {
        'TP': TP_cat,
        'cTP': cTP_total,
        'FP': FP_cat,
        'FN': FN_cat,
        'cPrecision': cPrecision,
        'cRecall': cRecall,
        'cF1': cF1
    }

# Subtask 1: RMSE Evaluation

In [16]:
# Laptop Domain - Task 1
gold_laptop_1 = read_jsonl('eng_laptop_test_task1.jsonl', task=1, data_type='gold')
pred_laptop_1 = read_jsonl('1/laptop_test_predictions.jsonl', task=1, data_type='pred')
results_laptop_1 = evaluate_task1(gold_laptop_1, pred_laptop_1, is_norm=True)

print("Laptop Domain - Task 1:")
print(f"PCC_V: {results_laptop_1['PCC_V']:.4f}")
print(f"PCC_A: {results_laptop_1['PCC_A']:.4f}")
print(f"RMSE_VA (normalized): {results_laptop_1['RMSE_VA']:.4f}\n")

Laptop Domain - Task 1:
PCC_V: 0.8024
PCC_A: 0.5032
RMSE_VA (normalized): 0.1311



In [17]:
# Restaurant Domain - Task 1
gold_restaurant_1 = read_jsonl('eng_restaurant_test_task1.jsonl', task=1, data_type='gold')
pred_restaurant_1 = read_jsonl('1/restaurant_test_predictions.jsonl', task=1, data_type='pred')
results_restaurant_1 = evaluate_task1(gold_restaurant_1, pred_restaurant_1, is_norm=True)

print("Restaurant Domain - Task 1:")
print(f"PCC_V: {results_restaurant_1['PCC_V']:.4f}")
print(f"PCC_A: {results_restaurant_1['PCC_A']:.4f}")
print(f"RMSE_VA (normalized): {results_restaurant_1['RMSE_VA']:.4f}\n")

Restaurant Domain - Task 1:
PCC_V: 0.8241
PCC_A: 0.5656
RMSE_VA (normalized): 0.1269



In [18]:
# Laptop Domain - Task 1
gold_laptop_1 = read_jsonl('eng_laptop_test_task1.jsonl', task=1, data_type='gold')
pred_laptop_1 = read_jsonl('1/laptop_test_predictions2.jsonl', task=1, data_type='pred')
results_laptop_1 = evaluate_task1(gold_laptop_1, pred_laptop_1, is_norm=True)

print("Laptop Domain - Task 1:")
print(f"PCC_V: {results_laptop_1['PCC_V']:.4f}")
print(f"PCC_A: {results_laptop_1['PCC_A']:.4f}")
print(f"RMSE_VA (normalized): {results_laptop_1['RMSE_VA']:.4f}\n")

Laptop Domain - Task 1:
PCC_V: 0.8079
PCC_A: 0.4882
RMSE_VA (normalized): 0.1283



In [22]:
# Restaurant Domain - Task 1
gold_restaurant_1 = read_jsonl('eng_restaurant_test_task1.jsonl', task=1, data_type='gold')
pred_restaurant_1 = read_jsonl('1/restaurant_test_predictions2.jsonl', task=1, data_type='pred')
results_restaurant_1 = evaluate_task1(gold_restaurant_1, pred_restaurant_1, is_norm=True)

print("Restaurant Domain - Task 1:")
print(f"PCC_V: {results_restaurant_1['PCC_V']:.4f}")
print(f"PCC_A: {results_restaurant_1['PCC_A']:.4f}")
print(f"RMSE_VA (normalized): {results_restaurant_1['RMSE_VA']:.4f}\n")

Restaurant Domain - Task 1:
PCC_V: 0.8253
PCC_A: 0.5541
RMSE_VA (normalized): 0.1256



In [19]:
# Laptop Domain - Task 1
gold_laptop_1 = read_jsonl('eng_laptop_test_task1.jsonl', task=1, data_type='gold')
pred_laptop_1 = read_jsonl('1/laptop_test_predictions3.jsonl', task=1, data_type='pred')
results_laptop_1 = evaluate_task1(gold_laptop_1, pred_laptop_1, is_norm=True)

print("Laptop Domain - Task 1:")
print(f"PCC_V: {results_laptop_1['PCC_V']:.4f}")
print(f"PCC_A: {results_laptop_1['PCC_A']:.4f}")
print(f"RMSE_VA (normalized): {results_laptop_1['RMSE_VA']:.4f}\n")

Laptop Domain - Task 1:
PCC_V: 0.8022
PCC_A: 0.4720
RMSE_VA (normalized): 0.1253



In [23]:
# Restaurant Domain - Task 1
gold_restaurant_1 = read_jsonl('eng_restaurant_test_task1.jsonl', task=1, data_type='gold')
pred_restaurant_1 = read_jsonl('1/restaurant_test_predictions3.jsonl', task=1, data_type='pred')
results_restaurant_1 = evaluate_task1(gold_restaurant_1, pred_restaurant_1, is_norm=True)

print("Restaurant Domain - Task 1:")
print(f"PCC_V: {results_restaurant_1['PCC_V']:.4f}")
print(f"PCC_A: {results_restaurant_1['PCC_A']:.4f}")
print(f"RMSE_VA (normalized): {results_restaurant_1['RMSE_VA']:.4f}\n")

Restaurant Domain - Task 1:
PCC_V: 0.8551
PCC_A: 0.5757
RMSE_VA (normalized): 0.1124



In [21]:
# Laptop Domain - Task 1
gold_laptop_1 = read_jsonl('eng_laptop_test_task1.jsonl', task=1, data_type='gold')
pred_laptop_1 = read_jsonl('1/laptop_test_predictions 4.jsonl', task=1, data_type='pred')
results_laptop_1 = evaluate_task1(gold_laptop_1, pred_laptop_1, is_norm=True)

print("Laptop Domain - Task 1:")
print(f"PCC_V: {results_laptop_1['PCC_V']:.4f}")
print(f"PCC_A: {results_laptop_1['PCC_A']:.4f}")
print(f"RMSE_VA (normalized): {results_laptop_1['RMSE_VA']:.4f}\n")

Laptop Domain - Task 1:
PCC_V: 0.7937
PCC_A: 0.4789
RMSE_VA (normalized): 0.1317



In [26]:
# Restaurant Domain - Task 1
gold_restaurant_1 = read_jsonl('eng_restaurant_test_task1.jsonl', task=1, data_type='gold')
pred_restaurant_1 = read_jsonl('1/restaurant_test_predictions4.jsonl', task=1, data_type='pred')
results_restaurant_1 = evaluate_task1(gold_restaurant_1, pred_restaurant_1, is_norm=True)

print("Restaurant Domain - Task 1:")
print(f"PCC_V: {results_restaurant_1['PCC_V']:.4f}")
print(f"PCC_A: {results_restaurant_1['PCC_A']:.4f}")
print(f"RMSE_VA (normalized): {results_restaurant_1['RMSE_VA']:.4f}\n")

Restaurant Domain - Task 1:
PCC_V: 0.8216
PCC_A: 0.5582
RMSE_VA (normalized): 0.1269



# Subtask 2: cF1 Evaluation

In [27]:
# Laptop Domain - Task 2
gold_laptop_2 = read_jsonl('eng_laptop_test_task2.jsonl', task=2, data_type='gold')
pred_laptop_2 = read_jsonl('2/laptop_test_predictions.jsonl', task=2, data_type='pred')
results_laptop_2 = evaluate_task23(gold_laptop_2, pred_laptop_2, task=2)

print("Laptop Domain - Task 2:")
print(f"TP: {results_laptop_2['TP']}")
print(f"cTP: {results_laptop_2['cTP']:.4f}")
print(f"FP: {results_laptop_2['FP']}")
print(f"FN: {results_laptop_2['FN']}")
print(f"cPrecision: {results_laptop_2['cPrecision']:.4f}")
print(f"cRecall: {results_laptop_2['cRecall']:.4f}")
print(f"cF1: {results_laptop_2['cF1']:.4f}\n")

Laptop Domain - Task 2:
TP: 875
cTP: 789.5710
FP: 536
FN: 1099
cPrecision: 0.5596
cRecall: 0.4000
cF1: 0.4665



In [28]:
# Restaurant Domain - Task 2
gold_restaurant_2 = read_jsonl('eng_restaurant_test_task2.jsonl', task=2, data_type='gold')
pred_restaurant_2 = read_jsonl('2/restaurant_test_predictions.jsonl', task=2, data_type='pred')
results_restaurant_2 = evaluate_task23(gold_restaurant_2, pred_restaurant_2, task=2)

print("Restaurant Domain - Task 2:")
print(f"TP: {results_restaurant_2['TP']}")
print(f"cTP: {results_restaurant_2['cTP']:.4f}")
print(f"FP: {results_restaurant_2['FP']}")
print(f"FN: {results_restaurant_2['FN']}")
print(f"cPrecision: {results_restaurant_2['cPrecision']:.4f}")
print(f"cRecall: {results_restaurant_2['cRecall']:.4f}")
print(f"cF1: {results_restaurant_2['cF1']:.4f}\n")

Restaurant Domain - Task 2:
TP: 855
cTP: 781.9680
FP: 468
FN: 1274
cPrecision: 0.5911
cRecall: 0.3673
cF1: 0.4531



In [29]:
# Laptop Domain - Task 2
gold_laptop_2 = read_jsonl('eng_laptop_test_task2.jsonl', task=2, data_type='gold')
pred_laptop_2 = read_jsonl('2/laptop_test_predictions2.jsonl', task=2, data_type='pred')
results_laptop_2 = evaluate_task23(gold_laptop_2, pred_laptop_2, task=2)

print("Laptop Domain - Task 2:")
print(f"TP: {results_laptop_2['TP']}")
print(f"cTP: {results_laptop_2['cTP']:.4f}")
print(f"FP: {results_laptop_2['FP']}")
print(f"FN: {results_laptop_2['FN']}")
print(f"cPrecision: {results_laptop_2['cPrecision']:.4f}")
print(f"cRecall: {results_laptop_2['cRecall']:.4f}")
print(f"cF1: {results_laptop_2['cF1']:.4f}\n")

Laptop Domain - Task 2:
TP: 818
cTP: 749.9404
FP: 444
FN: 1156
cPrecision: 0.5942
cRecall: 0.3799
cF1: 0.4635



In [31]:
# Restaurant Domain - Task 2
gold_restaurant_2 = read_jsonl('eng_restaurant_test_task2.jsonl', task=2, data_type='gold')
pred_restaurant_2 = read_jsonl('2/restaurant_test_predictions2.jsonl', task=2, data_type='pred')
results_restaurant_2 = evaluate_task23(gold_restaurant_2, pred_restaurant_2, task=2)

print("Restaurant Domain - Task 2:")
print(f"TP: {results_restaurant_2['TP']}")
print(f"cTP: {results_restaurant_2['cTP']:.4f}")
print(f"FP: {results_restaurant_2['FP']}")
print(f"FN: {results_restaurant_2['FN']}")
print(f"cPrecision: {results_restaurant_2['cPrecision']:.4f}")
print(f"cRecall: {results_restaurant_2['cRecall']:.4f}")
print(f"cF1: {results_restaurant_2['cF1']:.4f}\n")

Restaurant Domain - Task 2:
TP: 1142
cTP: 1041.8365
FP: 431
FN: 987
cPrecision: 0.6623
cRecall: 0.4894
cF1: 0.5629



In [30]:
# Laptop Domain - Task 2
gold_laptop_2 = read_jsonl('eng_laptop_test_task2.jsonl', task=2, data_type='gold')
pred_laptop_2 = read_jsonl('2/laptop_test_predictions3.jsonl', task=2, data_type='pred')
results_laptop_2 = evaluate_task23(gold_laptop_2, pred_laptop_2, task=2)

print("Laptop Domain - Task 2:")
print(f"TP: {results_laptop_2['TP']}")
print(f"cTP: {results_laptop_2['cTP']:.4f}")
print(f"FP: {results_laptop_2['FP']}")
print(f"FN: {results_laptop_2['FN']}")
print(f"cPrecision: {results_laptop_2['cPrecision']:.4f}")
print(f"cRecall: {results_laptop_2['cRecall']:.4f}")
print(f"cF1: {results_laptop_2['cF1']:.4f}\n")

Laptop Domain - Task 2:
TP: 851
cTP: 780.3946
FP: 436
FN: 1123
cPrecision: 0.6064
cRecall: 0.3953
cF1: 0.4786



In [32]:
# Restaurant Domain - Task 2
gold_restaurant_2 = read_jsonl('eng_restaurant_test_task2.jsonl', task=2, data_type='gold')
pred_restaurant_2 = read_jsonl('2/restaurant_test_predictions3.jsonl', task=2, data_type='pred')
results_restaurant_2 = evaluate_task23(gold_restaurant_2, pred_restaurant_2, task=2)

print("Restaurant Domain - Task 2:")
print(f"TP: {results_restaurant_2['TP']}")
print(f"cTP: {results_restaurant_2['cTP']:.4f}")
print(f"FP: {results_restaurant_2['FP']}")
print(f"FN: {results_restaurant_2['FN']}")
print(f"cPrecision: {results_restaurant_2['cPrecision']:.4f}")
print(f"cRecall: {results_restaurant_2['cRecall']:.4f}")
print(f"cF1: {results_restaurant_2['cF1']:.4f}\n")

Restaurant Domain - Task 2:
TP: 1151
cTP: 1057.9172
FP: 437
FN: 978
cPrecision: 0.6662
cRecall: 0.4969
cF1: 0.5692



# Subtask 3: cF1 Evaluation

In [33]:
# Laptop Domain - Task 3
gold_laptop_3 = read_jsonl('eng_laptop_test_task3.jsonl', task=3, data_type='gold')
pred_laptop_3 = read_jsonl('3/laptop_test_predictions1.jsonl', task=3, data_type='pred')
results_laptop_3 = evaluate_task23(gold_laptop_3, pred_laptop_3, task=3)

print("Laptop Domain - Task 3:")
print(f"TP: {results_laptop_3['TP']}")
print(f"cTP: {results_laptop_3['cTP']:.4f}")
print(f"FP: {results_laptop_3['FP']}")
print(f"FN: {results_laptop_3['FN']}")
print(f"cPrecision: {results_laptop_3['cPrecision']:.4f}")
print(f"cRecall: {results_laptop_3['cRecall']:.4f}")
print(f"cF1: {results_laptop_3['cF1']:.4f}\n")

Laptop Domain - Task 3:
TP: 0
cTP: 0.0000
FP: 1822
FN: 1975
cPrecision: 0.0000
cRecall: 0.0000
cF1: 0.0000



In [34]:
# Restaurant Domain - Task 3
gold_restaurant_3 = read_jsonl('eng_restaurant_test_task3.jsonl', task=3, data_type='gold')
pred_restaurant_3 = read_jsonl('3/restaurant_test_predictions1.jsonl', task=3, data_type='pred')
results_restaurant_3 = evaluate_task23(gold_restaurant_3, pred_restaurant_3, task=3)

print("Restaurant Domain - Task 3:")
print(f"TP: {results_restaurant_3['TP']}")
print(f"cTP: {results_restaurant_3['cTP']:.4f}")
print(f"FP: {results_restaurant_3['FP']}")
print(f"FN: {results_restaurant_3['FN']}")
print(f"cPrecision: {results_restaurant_3['cPrecision']:.4f}")
print(f"cRecall: {results_restaurant_3['cRecall']:.4f}")
print(f"cF1: {results_restaurant_3['cF1']:.4f}\n")

Restaurant Domain - Task 3:
TP: 1100
cTP: 1020.0321
FP: 1031
FN: 1029
cPrecision: 0.4787
cRecall: 0.4791
cF1: 0.4789



In [35]:
# Laptop Domain - Task 3
gold_laptop_3 = read_jsonl('eng_laptop_test_task3.jsonl', task=3, data_type='gold')
pred_laptop_3 = read_jsonl('3/laptop_test_predictions2.jsonl', task=3, data_type='pred')
results_laptop_3 = evaluate_task23(gold_laptop_3, pred_laptop_3, task=3)

print("Laptop Domain - Task 3:")
print(f"TP: {results_laptop_3['TP']}")
print(f"cTP: {results_laptop_3['cTP']:.4f}")
print(f"FP: {results_laptop_3['FP']}")
print(f"FN: {results_laptop_3['FN']}")
print(f"cPrecision: {results_laptop_3['cPrecision']:.4f}")
print(f"cRecall: {results_laptop_3['cRecall']:.4f}")
print(f"cF1: {results_laptop_3['cF1']:.4f}\n")

Laptop Domain - Task 3:
TP: 525
cTP: 485.4928
FP: 1182
FN: 1450
cPrecision: 0.2844
cRecall: 0.2458
cF1: 0.2637



In [36]:
# Restaurant Domain - Task 3
gold_restaurant_3 = read_jsonl('eng_restaurant_test_task3.jsonl', task=3, data_type='gold')
pred_restaurant_3 = read_jsonl('3/restaurant_test_predictions2.jsonl', task=3, data_type='pred')
results_restaurant_3 = evaluate_task23(gold_restaurant_3, pred_restaurant_3, task=3)

print("Restaurant Domain - Task 3:")
print(f"TP: {results_restaurant_3['TP']}")
print(f"cTP: {results_restaurant_3['cTP']:.4f}")
print(f"FP: {results_restaurant_3['FP']}")
print(f"FN: {results_restaurant_3['FN']}")
print(f"cPrecision: {results_restaurant_3['cPrecision']:.4f}")
print(f"cRecall: {results_restaurant_3['cRecall']:.4f}")
print(f"cF1: {results_restaurant_3['cF1']:.4f}\n")

Restaurant Domain - Task 3:
TP: 1158
cTP: 1066.6256
FP: 740
FN: 971
cPrecision: 0.5620
cRecall: 0.5010
cF1: 0.5297



In [37]:
# Laptop Domain - Task 3
gold_laptop_3 = read_jsonl('eng_laptop_test_task3.jsonl', task=3, data_type='gold')
pred_laptop_3 = read_jsonl('3/laptop_test_predictions3.jsonl', task=3, data_type='pred')
results_laptop_3 = evaluate_task23(gold_laptop_3, pred_laptop_3, task=3)

print("Laptop Domain - Task 3:")
print(f"TP: {results_laptop_3['TP']}")
print(f"cTP: {results_laptop_3['cTP']:.4f}")
print(f"FP: {results_laptop_3['FP']}")
print(f"FN: {results_laptop_3['FN']}")
print(f"cPrecision: {results_laptop_3['cPrecision']:.4f}")
print(f"cRecall: {results_laptop_3['cRecall']:.4f}")
print(f"cF1: {results_laptop_3['cF1']:.4f}\n")

Laptop Domain - Task 3:
TP: 548
cTP: 505.4485
FP: 1159
FN: 1427
cPrecision: 0.2961
cRecall: 0.2559
cF1: 0.2746



In [38]:
# Restaurant Domain - Task 3
gold_restaurant_3 = read_jsonl('eng_restaurant_test_task3.jsonl', task=3, data_type='gold')
pred_restaurant_3 = read_jsonl('3/restaurant_test_predictions3.jsonl', task=3, data_type='pred')
results_restaurant_3 = evaluate_task23(gold_restaurant_3, pred_restaurant_3, task=3)

print("Restaurant Domain - Task 3:")
print(f"TP: {results_restaurant_3['TP']}")
print(f"cTP: {results_restaurant_3['cTP']:.4f}")
print(f"FP: {results_restaurant_3['FP']}")
print(f"FN: {results_restaurant_3['FN']}")
print(f"cPrecision: {results_restaurant_3['cPrecision']:.4f}")
print(f"cRecall: {results_restaurant_3['cRecall']:.4f}")
print(f"cF1: {results_restaurant_3['cF1']:.4f}\n")

Restaurant Domain - Task 3:
TP: 1180
cTP: 1082.0362
FP: 734
FN: 949
cPrecision: 0.5653
cRecall: 0.5082
cF1: 0.5353

